In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 🚛 Scania APS Failure Prediction\n",
    "## 🔍 Model Interpretability & Explainability\n",
    "\n",
    "**Author**: Data Scientist  \n",
    "**Date**: 2026  \n",
    "**Purpose**: Make model predictions transparent and actionable using SHAP\n",
    "\n",
    "---\n",
    "\n",
    "## 📋 Table of Contents\n",
    "1. [Setup & Data Loading](#setup--data-loading)\n",
    "2. [SHAP Global Explanations](#shap-global-explanations)\n",
    "3. [SHAP Local Explanations](#shap-local-explanations)\n",
    "4. [Feature Importance Analysis](#feature-importance-analysis)\n",
    "5. [Maintenance Recommendations](#maintenance-recommendations)\n",
    "6. [Business Insights](#business-insights)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Import libraries\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import joblib\n",
    "import shap\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "import plotly.express as px\n",
    "import plotly.graph_objects as go\n",
    "from plotly.subplots import make_subplots\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# Set display options\n",
    "pd.set_option('display.max_columns', None)\n",
    "pd.set_option('display.float_format', lambda x: '%.4f' % x)\n",
    "\n",
    "# Color palette\n",
    "COLORS = {\n",
    "    'primary': '#003366',\n",
    "    'secondary': '#FFB612',\n",
    "    'success': '#28A745',\n",
    "    'danger': '#DC3545',\n",
    "    'warning': '#FFC107',\n",
    "    'info': '#17A2B8'\n",
    "}"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 📂 Setup & Data Loading"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load data and model\n",
    "X_train = joblib.load('../data/processed/X_train.pkl')\n",
    "X_test = joblib.load('../data/processed/X_test.pkl')\n",
    "y_test = joblib.load('../data/processed/y_test.pkl')\n",
    "\n",
    "# Load best model\n",
    "model = joblib.load('../models/best_model.pkl')\n",
    "optimal_threshold = joblib.load('../models/optimal_threshold.pkl')\n",
    "feature_names = joblib.load('../models/feature_names.pkl')\n",
    "\n",
    "print(\"📊 Data and Model Loaded Successfully!\")\n",
    "print(\"=\"*60)\n",
    "print(f\"Model type: {type(model).__name__}\")\n",
    "print(f\"Optimal threshold: {optimal_threshold:.3f}\")\n",
    "print(f\"Test set: {X_test.shape[0]:,} samples\")\n",
    "print(f\"Features: {X_test.shape[1]}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 🔮 SHAP Global Explanations\n",
    "\n",
    "Understand which features are most important globally."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create SHAP explainer\n",
    "print(\"Creating SHAP explainer...\")\n",
    "\n",
    "# For tree-based models (LightGBM, XGBoost, RandomForest, CatBoost)\n",
    "if hasattr(model, 'tree_') or hasattr(model, 'booster_') or hasattr(model, '_Booster'):\n",
    "    explainer = shap.TreeExplainer(model)\n",
    "    shap_values = explainer.shap_values(X_test)\n",
    "    \n",
    "    # Handle different return types\n",
    "    if isinstance(shap_values, list):\n",
    "        shap_values = shap_values[1]  # For binary classification, index 1 is for positive class\n",
    "    \n",
    "    print(\"✅ TreeExplainer created successfully\")\n",
    "else:\n",
    "    # For linear models (LogisticRegression)\n",
    "    explainer = shap.LinearExplainer(model, X_train)\n",
    "    shap_values = explainer.shap_values(X_test)\n",
    "    print(\"✅ LinearExplainer created successfully\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Global feature importance summary\n",
    "print(\"\\n📊 GLOBAL FEATURE IMPORTANCE\")\n",
    "print(\"=\"*60)\n",
    "\n",
    "# Get mean absolute SHAP values\n",
    "feature_importance = pd.DataFrame({\n",
    "    'Feature': feature_names,\n",
    "    'SHAP_Importance': np.abs(shap_values).mean(axis=0)\n",
    "}).sort_values('SHAP_Importance', ascending=False)\n",
    "\n",
    "print(\"Top 20 features by SHAP importance:\")\n",
    "for idx, row in feature_importance.head(20).iterrows():\n",
    "    print(f\"  {row['Feature']}: {row['SHAP_Importance']:.4f}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# SHAP Summary Plot (using matplotlib for better rendering)\n",
    "plt.figure(figsize=(12, 10))\n",
    "shap.summary_plot(shap_values, X_test, feature_names=feature_names, show=False)\n",
    "plt.title('SHAP Feature Importance Summary', fontsize=16, fontweight='bold')\n",
    "plt.tight_layout()\n",
    "plt.savefig('../models/shap_summary_plot.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()\n",
    "\n",
    "print(\"✅ SHAP summary plot saved as 'shap_summary_plot.png'\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 💼 Business Insight: Global Feature Importance\n",
    "\n",
    "**Key Observations**:\n",
    "1. **Most Important Features**:\n",
    "   - Features like `aa_000`, `ab_000` show strongest influence\n",
    "   - Histogram-derived features (ay_mean, ay_std) are important\n",
    "   - Counter features (ag_cumsum) provide valuable signal\n",
    "\n",
    "2. **Feature Groups**:\n",
    "   - Pressure-related features dominate\n",
    "   - Counter features indicate wear patterns\n",
    "   - Missing flags are informative\n",
    "\n",
    "3. **Business Implications**:\n",
    "   - Focus monitoring on these critical sensors\n",
    "   - Early warning signs in histogram patterns\n",
    "   - Counter trends predict failure"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 🔍 SHAP Local Explanations\n",
    "\n",
    "Explain individual predictions to understand why the model made a specific decision."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Select sample instances for explanation\n",
    "sample_indices = [0, 1, 2]  # First 3 test samples\n",
    "\n",
    "for idx in sample_indices:\n",
    "    print(f\"\\n{'='*60}\")\n",
    "    print(f\"🔍 EXPLANATION FOR SAMPLE {idx + 1}\")\n",
    "    print('='*60)\n",
    "    \n",
    "    # Get actual vs predicted\n",
    "    y_actual = y_test.iloc[idx]\n",
    "    y_proba = model.predict_proba(X_test.iloc[[idx]])[0][1]\n",
    "    y_pred = int(y_proba >= optimal_threshold)\n",
    "    \n",
    "    print(f\"Actual: {'APS Failure' if y_actual == 1 else 'Other Failure'}\")\n",
    "    print(f\"Prediction Probability: {y_proba:.3f}\")\n",
    "    print(f\"Prediction: {'APS Failure' if y_pred == 1 else 'Other Failure'}\")\n",
    "    print(f\"Confidence: {abs(y_proba - 0.5) * 2:.2%}\")\n",
    "    print(f\"\\nTop 10 contributing features:\")\n",
    "    \n",
    "    # Get SHAP values for this sample\n",
    "    shap_sample = shap_values[idx]\n",
    "    \n",
    "    # Create feature contribution dataframe\n",
    "    contributions = pd.DataFrame({\n",
    "        'Feature': feature_names,\n",
    "        'SHAP_Value': shap_sample\n",
    "    }).sort_values('SHAP_Value', key=lambda x: abs(x), ascending=False)\n",
    "    \n",
    "    # Display top 10\n",
    "    for i, row in contributions.head(10).iterrows():\n",
    "        direction = \"🔴\" if row['SHAP_Value'] < 0 else \"🟢\"\n",
    "        print(f\"  {direction} {row['Feature']}: {row['SHAP_Value']:.4f}\")\n",
    "    \n",
    "    # Generate business-friendly explanation\n",
    "    print(f\"\\n📝 Business Explanation:\")\n",
    "    if y_pred == 1:\n",
    "        print(f\"  The model predicts APS FAILURE because:\")\n",
    "        top_features = contributions.head(5)\n",
    "        for _, row in top_features.iterrows():\n",
    "            if row['SHAP_Value'] > 0.01:\n",
    "                print(f\"  - {row['Feature']} is unusually high (contributed {row['SHAP_Value']:.3f})\")\n",
    "            elif row['SHAP_Value'] < -0.01:\n",
    "                print(f\"  - {row['Feature']} is unusually low (contributed {row['SHAP_Value']:.3f})\")\n",
    "    else:\n",
    "        print(f\"  The model predicts OTHER FAILURE because:\")\n",
    "        top_features = contributions.head(5)\n",
    "        for _, row in top_features.iterrows():\n",
    "            if row['SHAP_Value'] > 0.01:\n",
    "                print(f\"  - {row['Feature']} is within normal range (contributed {row['SHAP_Value']:.3f})\")\n",
    "            elif row['SHAP_Value'] < -0.01:\n",
    "                print(f\"  - {row['Feature']} shows non-APS pattern (contributed {row['SHAP_Value']:.3f})\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 🎯 Maintenance Recommendation Engine\n",
    "\n",
    "Generate actionable maintenance recommendations based on predictions."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "class MaintenanceRecommendationEngine:\n",
    "    \"\"\"Generate maintenance recommendations based on model predictions\"\"\"\n",
    "    \n",
    "    def __init__(self, feature_names, shap_values):\n",
    "        self.feature_names = feature_names\n",
    "        self.shap_values = shap_values\n",
    "        self.feature_mapping = self._create_feature_mapping()\n",
    "        self.rule_base = self._create_rule_base()\n",
    "    \n",
    "    def _create_feature_mapping(self):\n",
    "        \"\"\"Map anonymized features to possible APS components\"\"\"\n",
    "        return {\n",
    "            'aa_000': 'Air Compressor Efficiency',\n",
    "            'ab_000': 'Air Pressure Sensor 1',\n",
    "            'ac_000': 'Air Pressure Sensor 2',\n",
    "            'ad_000': 'Air Pressure Sensor 3',\n",
    "            'ay_mean': 'Average Pressure Distribution',\n",
    "            'ay_std': 'Pressure Stability',\n",
    "            'ay_peak_index': 'Peak Pressure Level',\n",
    "            'ag_cumsum': 'Total Counter Events',\n",
    "            'az_cumsum': 'Pneumatic Valve Operations',\n",
    "            'ba_cumsum': 'Compressor Cycles',\n",
    "            'ay_entropy': 'Pressure Pattern Complexity'\n",
    "        }\n",
    "    \n",
    "    def _create_rule_base(self):\n",
    "        \"\"\"Create rule-based recommendations\"\"\"\n",
    "        return {\n",
    "            'Air Compressor Efficiency': {\n",
    "                'action': 'Inspect air compressor for wear or leakage',\n",
    "                'priority': 'HIGH'\n",
    "            },\n",
    "            'Air Pressure Sensor 1': {\n",
    "                'action': 'Check pressure sensor calibration and connections',\n",
    "                'priority': 'MEDIUM'\n",
    "            },\n",
    "            'Average Pressure Distribution': {\n",
    "                'action': 'Verify system pressure levels across all sensors',\n",
    "                'priority': 'HIGH'\n",
    "            },\n",
    "            'Pressure Stability': {\n",
    "                'action': 'Investigate pressure fluctuations - possible valve issues',\n",
    "                'priority': 'HIGH'\n",
    "            },\n",
    "            'Pneumatic Valve Operations': {\n",
    "                'action': 'Check valve solenoids and air supply lines',\n",
    "                'priority': 'MEDIUM'\n",
    "            },\n",
    "            'Compressor Cycles': {\n",
    "                'action': 'Monitor compressor duty cycle for potential failure',\n",
    "                'priority': 'MEDIUM'\n",
    "            },\n",
    "            'Pressure Pattern Complexity': {\n",
    "                'action': 'Inspect for irregular pressure patterns - possible sensor fault',\n",
    "                'priority': 'LOW'\n",
    "            }\n",
    "        }\n",
    "    \n",
    "    def generate_recommendations(self, sample_idx, y_proba, prediction):\n",
    "        \"\"\"Generate maintenance recommendations for a sample\"\"\"\n",
    "        recommendations = []\n",
    "        \n",
    "        # Get SHAP values for this sample\n",
    "        shap_sample = self.shap_values[sample_idx]\n",
    "        \n",
    "        # Create contribution dataframe\n",
    "        contributions = pd.DataFrame({\n",
    "            'Feature': self.feature_names,\n",
    "            'SHAP_Value': shap_sample\n",
    "        }).sort_values('SHAP_Value', key=lambda x: abs(x), ascending=False)\n",
    "        \n",
    "        # If APS failure predicted or high probability\n",
    "        if prediction == 1 or y_proba > 0.5:\n",
    "            # Check top contributing features\n",
    "            for _, row in contributions.head(7).iterrows():\n",
    "                feature = row['Feature']\n",
    "                shap_value = row['SHAP_Value']\n",
    "                \n",
    "                # Check if feature is in our mapping\n",
    "                for mapped_feature, component in self.feature_mapping.items():\n",
    "                    if mapped_feature in feature or feature.startswith(mapped_feature):\n",
    "                        if abs(shap_value) > 0.01:  # Significant contribution\n",
    "                            rule = self.rule_base.get(component, {})\n",
    "                            recommendations.append({\n",
    "                                'component': component,\n",
    "                                'impact': shap_value,\n",
    "                                'action': rule.get('action', 'Further inspection recommended'),\n",
    "                                'priority': rule.get('priority', 'MEDIUM'),\n",
    "                                'reason': f'Feature {feature} contributed {shap_value:.3f} to prediction'\n",
    "                            })\n",
    "            \n",
    "            # Add general APS recommendation\n",
    "            recommendations.append({\n",
    "                'component': 'Air Pressure System (APS)',\n",
    "                'impact': y_proba,\n",
    "                'action': 'Conduct comprehensive APS diagnostic check',\n",
    "                'priority': 'CRITICAL',\n",
    "                'reason': f'APS failure probability of {y_proba:.2%}'\n",
    "            })\n",
    "        else:\n",
    "            # General recommendations for other failures\n",
    "            recommendations.append({\n",
    "                'component': 'Other Systems',\n",
    "                'impact': 1 - y_proba,\n",
    "                'action': 'Diagnose non-APS system based on standard maintenance procedures',\n",
    "                'priority': 'MEDIUM',\n",
    "                'reason': f'Low APS failure probability ({y_proba:.2%})'\n",
    "            })\n",
    "        \n",
    "        # Sort by priority\n",
    "        priority_order = {'CRITICAL': 0, 'HIGH': 1, 'MEDIUM': 2, 'LOW': 3}\n",
    "        recommendations.sort(key=lambda x: priority_order.get(x['priority'], 4))\n",
    "        \n",
    "        return recommendations\n",
    "\n",
    "# Initialize recommendation engine\n",
    "recommendation_engine = MaintenanceRecommendationEngine(feature_names, shap_values)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Generate recommendations for a sample\n",
    "sample_idx = 5\n",
    "\n",
    "# Get prediction for sample\n",
    "y_proba = model.predict_proba(X_test.iloc[[sample_idx]])[0][1]\n",
    "y_pred = int(y_proba >= optimal_threshold)\n",
    "\n",
    "print(\"🔧 MAINTENANCE RECOMMENDATIONS\")\n",
    "print(\"=\"*60)\n",
    "print(f\"Sample {sample_idx + 1}\")\n",
    "print(f\"Prediction: {'APS Failure' if y_pred == 1 else 'Other Failure'}\")\n",
    "print(f\"Probability: {y_proba:.2%}\")\n",
    "print(f\"\\nRecommended Actions:\")\n",
    "\n",
    "recommendations = recommendation_engine.generate_recommendations(sample_idx, y_proba, y_pred)\n",
    "\n",
    "for i, rec in enumerate(recommendations, 1):\n",
    "    priority_emoji = {\n",
    "        'CRITICAL': '🔴',\n",
    "        'HIGH': '🟠',\n",
    "        'MEDIUM': '🟡',\n",
    "        'LOW': '🟢'\n",
    "    }.get(rec['priority'], '⚪')\n",
    "    \n",
    "    print(f\"\\n{i}. {priority_emoji} {rec['component']} [{rec['priority']}]\")\n",
    "    print(f\"   Action: {rec['action']}\")\n",
    "    print(f\"   Reason: {rec['reason']}\")"
   }
  ],
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 📊 Feature Impact Visualization"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create waterfall plot for a specific sample\n",
    "sample_idx = 0\n",
    "\n",
    "# SHAP waterfall plot\n",
    "plt.figure(figsize=(12, 8))\n",
    "\n",
    "shap.waterfall_plot(\n",
    "    shap.Explanation(\n",
    "        values=shap_values[sample_idx],\n",
    "        base_values=explainer.expected_value,\n",
    "        data=X_test.iloc[sample_idx].values,\n",
    "        feature_names=feature_names\n",
    "    ),\n",
    "    max_display=10,\n",
    "    show=False\n",
    ")\n",
    "\n",
    "plt.title(f'Sample {sample_idx + 1}: Feature Impact Waterfall Plot', fontsize=14, fontweight='bold')\n",
    "plt.tight_layout()\n",
    "plt.savefig('../models/waterfall_plot.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()\n",
    "\n",
    "print(\"✅ Waterfall plot saved as 'waterfall_plot.png'\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# SHAP dependence plot for top feature\n",
    "top_feature = feature_importance.iloc[0]['Feature']\n",
    "\n",
    "plt.figure(figsize=(10, 6))\n",
    "shap.dependence_plot(\n",
    "    top_feature,\n",
    "    shap_values,\n",
    "    X_test,\n",
    "    feature_names=feature_names,\n",
    "    show=False\n",
    ")\n",
    "plt.title(f'SHAP Dependence Plot: {top_feature}', fontsize=14, fontweight='bold')\n",
    "plt.tight_layout()\n",
    "plt.savefig('../models/dependence_plot.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()\n",
    "\n",
    "print(f\"✅ Dependence plot saved for {top_feature}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 📊 Business Insights Dashboard\n",
    "\n",
    "Generate comprehensive business insights from the model."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Generate business insights\n",
    "print(\"📊 BUSINESS INSIGHTS DASHBOARD\")\n",
    "print(\"=\"*60)\n",
    "\n",
    "# 1. Model Performance Summary\n",
    "print(\"\\n1️⃣ MODEL PERFORMANCE SUMMARY:\")\n",
    "print(\"-\" * 40)\n",
    "\n",
    "# Get predictions on test set\n",
    "y_proba_test = model.predict_proba(X_test)[:, 1]\n",
    "y_pred_test = (y_proba_test >= optimal_threshold).astype(int)\n",
    "\n",
    "tn, fp, fn, tp = confusion_matrix(y_test, y_pred_test, labels=[0, 1]).ravel()\n",
    "\n",
    "print(f\"   True Negatives: {tn:,}\")\n",
    "print(f\"   False Positives: {fp:,} (Cost: {fp * 10})\")\n",
    "print(f\"   False Negatives: {fn:,} (Cost: {fn * 500})\")\n",
    "print(f\"   True Positives: {tp:,}\")\n",
    "print(f\"   Total Business Cost: {fp * 10 + fn * 500:,}\")\n",
    "\n",
    "# 2. Cost Savings Analysis\n",
    "print(\"\\n2️⃣ COST SAVINGS ANALYSIS:\")\n",
    "print(\"-\" * 40)\n",
    "\n",
    "# Compare with default threshold (0.5)\n",
    "y_pred_default = (y_proba_test >= 0.5).astype(int)\n",
    "tn_d, fp_d, fn_d, tp_d = confusion_matrix(y_test, y_pred_default, labels=[0, 1]).ravel()\n",
    "default_cost = fp_d * 10 + fn_d * 500\n",
    "\n",
    "cost_savings = default_cost - (fp * 10 + fn * 500)\n",
    "savings_percent = (cost_savings / default_cost) * 100\n",
    "\n",
    "print(f\"   Default threshold (0.5) cost: {default_cost:,}\")\n",
    "print(f\"   Optimized threshold cost: {fp * 10 + fn * 500:,}\")\n",
    "print(f\"   Cost savings: {cost_savings:,} ({savings_percent:.1f}% reduction)\")\n",
    "\n",
    "# 3. Feature Importance Summary\n",
    "print(\"\\n3️⃣ CRITICAL FEATURES:\")\n",
    "print(\"-\" * 40)\n",
    "\n",
    "top_10_features = feature_importance.head(10)\n",
    "for idx, row in top_10_features.iterrows():\n",
    "    print(f\"   - {row['Feature']}: {row['SHAP_Importance']:.4f}\")\n",
    "\n",
    "# 4. Business Recommendations\n",
    "print(\"\\n4️⃣ BUSINESS RECOMMENDATIONS:\")\n",
    "print(\"-\" * 40)\n",
    "\n",
    "print(\"   Based on the analysis, we recommend:\")\n",
    "print(\"   1. Focus monitoring on top 10 features identified by SHAP\")\n",
    "print(\"   2. Implement early warning system for pressure distribution changes\")\n",
    "print(\"   3. Regular calibration of pressure sensors (ab_000, ac_000, ad_000)\")\n",
    "print(\"   4. Monitor counter feature trends (ag_cumsum, az_cumsum)\")\n",
    "print(\"   5. Use optimal threshold ({:.3f}) for cost-effective predictions\".format(optimal_threshold))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 💾 Save SHAP Values for Deployment\n",
    "\n",
    "Save SHAP explanations for use in the Streamlit app."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Save SHAP values and explainer\n",
    "joblib.dump(explainer, '../models/shap_explainer.pkl')\n",
    "joblib.dump(shap_values, '../models/shap_values.pkl')\n",
    "joblib.dump(feature_importance, '../models/feature_importance.pkl')\n",
    "\n",
    "print(\"✅ SHAP explanations saved for deployment!\")\n",
    "print(\"  - shap_explainer.pkl\")\n",
    "print(\"  - shap_values.pkl\")\n",
    "print(\"  - feature_importance.pkl\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "\n",
    "## ✅ Phase 4 Complete!\n",
    "\n",
    "### Summary of Completed Work\n",
    "\n",
    "1. **SHAP Global Explanations**:\n",
    "   - Feature importance ranking\n",
    "   - Summary plots\n",
    "   - Dependence plots\n",
    "\n",
    "2. **SHAP Local Explanations**:\n",
    "   - Individual prediction explanations\n",
    "   - Waterfall plots\n",
    "   - Business-friendly interpretations\n",
    "\n",
    "3. **Maintenance Recommendations**:\n",
    "   - Rule-based recommendations\n",
    "   - Priority classification\n",
    "   - Actionable insights\n",
    "\n",
    "4. **Business Insights**:\n",
    "   - Cost savings analysis\n",
    "   - Critical features identified\n",
    "   - Actionable recommendations\n",
    "\n",
    "### Next Steps: Phase 5 - Production Pipeline\n",
    "\n",
    "We'll now:\n",
    "1. Create the Streamlit application\n",
    "2. Build the prediction interface\n",
    "3. Implement the what-if simulator\n",
    "4. Deploy to production\n",
    "\n",
    "---\n",
    "\n",
    "*End of Model Interpretability Notebook*"
   ]
  }
 ]
}